[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ronniewillaert/SPM-Textbook-Python/blob/main/notebooks/part-I-introduction/ch02/AFM_Force_Distance_LJ_Hertz_Adhesion.ipynb)

# 📘 AFM Force–Distance Simulation Notebook
**Lennard–Jones + Contact Mechanics + Adhesion + Hysteresis**

*Accompanying notebook for Chapter 2 — Scanning Probe Microscopy textbook*

In [ ]:
# ── Google Colab setup (run this cell first) ────────────────────────
import sys
if "google.colab" in sys.modules:
    !pip install ipywidgets -q
    from google.colab import output
    output.enable_custom_widget_manager()
    print("✅ Colab environment ready — widgets enabled.")
else:
    print("ℹ️  Running locally — ensure ipywidgets is installed.")

🔷 1. Introduction
In Atomic Force Microscopy (AFM), force–distance curves reflect the combined effects of:
Interatomic attraction (Lennard–Jones interaction)
Cantilever mechanics and stability
Elastic contact (Hertz model)
Adhesion (DMT/JKR-inspired models)
Energy dissipation (hysteresis)
In this notebook, we construct a unified computational framework that reproduces these regimes and allows quantitative analysis of:
Jump-to-contact
Pull-off force
Effective Young’s modulus
Dissipated energy

🔷 2. Import Required Libraries

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import brentq
import ipywidgets as widgets
from IPython.display import display, clear_output

🔷 3. Define Physical Interaction Models
3.1 Lennard–Jones Force

In [ ]:
def lennard_jones_force(r, epsilon, sigma):
    """Compute LJ force using ratio form (avoids overflow for small r)."""
    r = np.maximum(r, 0.5 * sigma)   # hard floor to prevent divergence
    sr = sigma / r
    sr6 = sr**6
    return (24.0 * epsilon / r) * (2.0 * sr6 * sr6 - sr6)

3.2 Hertz Contact Force

In [3]:
def hertz_force(delta, E_star, R):
    delta = np.maximum(delta, 0.0)
    return (4.0/3.0) * E_star * np.sqrt(R) * delta**1.5

3.3 Adhesion Offset (DMT / JKR-inspired)

In [4]:
def adhesive_offset(R, Wadh, model="DMT"):
    if model.upper() == "DMT":
        C = 2.0
    elif model.upper() == "JKR":
        C = 1.5
    else:
        raise ValueError("model must be 'DMT' or 'JKR'")
    return -C * np.pi * R * Wadh

🔷 4. Total Interaction Force
This combines:
- **Non-contact (D ≥ 0):** Lennard–Jones attraction/repulsion
- **Contact (D < 0):** Hertz elastic + adhesion + optional hysteresis
- Adhesion tapers linearly with indentation depth (→ 0 at the contact edge),
  ensuring a natural **snap-off** during retraction

The piecewise switch at D = 0 is solved exactly by `scipy.optimize.brentq`
(root-finding in each regime), so no relaxation artefacts can appear.

In [ ]:
def F_noncontact(D, params):
    """Non-contact force (D >= 0): Lennard-Jones."""
    r = D + params["r0"]
    return lennard_jones_force(r, params["epsilon"], params["sigma"])


def F_contact(D, params, branch="approach"):
    """Contact force (D < 0): Hertz + adhesion + hysteresis."""
    delta = -D   # indentation depth (positive)
    F_H = hertz_force(delta, params["E_star"], params["R"])

    F_adh_full = adhesive_offset(params["R"], params["Wadh"],
                                  model=params["adh_model"])
    # Adhesion scales with indentation: full adhesion beyond delta_ref,
    # tapers linearly to zero at the contact edge (delta = 0).
    # This ensures snap-off occurs naturally during retraction.
    delta_ref = params["sigma"]
    adh_scale = min(delta / delta_ref, 1.0) if delta > 0 else 0.0
    F_adh = F_adh_full * adh_scale

    if branch == "retract":
        F_hyst = -params["hyst_strength"] * np.tanh(delta / params["hyst_delta0"])
    else:
        F_hyst = 0.0

    return F_H + F_adh + F_hyst


def total_interaction_force(D, params, branch="approach"):
    """Piecewise force: LJ for D >= 0, Hertz+adhesion for D < 0."""
    if D >= 0:
        return F_noncontact(D, params)
    else:
        return F_contact(D, params, branch)

🔷 5. Coupled Cantilever Solver

We solve quasi-static equilibrium: $k \, x = F_{\mathrm{int}}(D)$ with $D = z_{\mathrm{piezo}} - x$.

Reformulated as a root problem in $D$:

$$g(D) \;=\; F_{\mathrm{int}}(D) + k\,D - k\,z \;=\; 0$$

Roots in the **non-contact** ($D > 0$) and **contact** ($D < 0$) regimes are
found independently with `brentq`.  During approach the solver follows the
non-contact branch until it vanishes (→ jump-to-contact); during retraction
it follows the contact branch until it vanishes (→ snap-off).

The measured cantilever force uses the standard AFM convention:
$$F = k\,(D - z)$$
where **positive** = repulsive (tip pushed away from sample) and **negative** = attractive (adhesion).

In [ ]:
def simulate_fd_curve(params, z_min=-30e-9, z_max=50e-9, n=500):
    """
    Simulate an AFM force–distance curve using exact root-finding.

    The equilibrium g(D) = F_int(D) + k·D − k·z = 0 is solved separately
    in the non-contact (D > 0, LJ) and contact (D < 0, Hertz+adhesion)
    regimes.  Branch tracking ensures physically correct jump-to-contact
    and snap-off.

    Returns
    -------
    (z_app, F_app), (z_ret, F_ret)
        Piezo position and force arrays for approach and retraction.
        Force convention: F = k·(D − z), positive = repulsive (standard AFM).
    """
    k = params["k"]
    z_approach = np.linspace(z_max, z_min, n)
    z_retract  = np.linspace(z_min, z_max, n)

    # ── helpers ────────────────────────────────────────────────────
    def _find_roots(func, D_lo, D_hi, z, extra_args=(), n_scan=300):
        """Scan [D_lo, D_hi] for sign changes, refine each with brentq."""
        if D_lo >= D_hi:
            return []
        D_scan = np.linspace(D_lo, D_hi, n_scan)
        g_scan = np.array([func(d, z, *extra_args) for d in D_scan])
        roots = []
        for idx in np.where(np.diff(np.sign(g_scan)))[0]:
            if np.isfinite(g_scan[idx]) and np.isfinite(g_scan[idx + 1]):
                try:
                    roots.append(
                        brentq(func, D_scan[idx], D_scan[idx + 1],
                               args=(z, *extra_args), xtol=1e-14, maxiter=100)
                    )
                except (ValueError, RuntimeError):
                    pass
        return sorted(roots)

    def g_nc(D, z):
        """Residual in the non-contact regime."""
        return F_noncontact(D, params) + k * D - k * z

    def g_c(D, z, branch_name):
        """Residual in the contact regime."""
        return F_contact(D, params, branch_name) + k * D - k * z

    # ── generic branch solver ─────────────────────────────────────
    def _solve(z_array, branch_name, D0):
        D_prev = D0
        on_primary = True          # approach → NC first; retract → C first
        is_approach = (branch_name == "approach")
        D_out, F_out = [], []

        for z in z_array:
            D_hi = max(z + 5e-9, z_max + 5e-9)

            if is_approach:
                prim_func, prim_range = g_nc, (1e-12, D_hi)
                sec_func,  sec_range  = g_c,  (-30e-9, -1e-12)
                prim_args, sec_args   = (), ("approach",)
            else:
                prim_func, prim_range = g_c,  (-30e-9, -1e-12)
                sec_func,  sec_range  = g_nc, (1e-12, D_hi)
                prim_args, sec_args   = ("retract",), ()

            if on_primary:
                # narrow search around D_prev, then wide fallback
                hw = 3e-9
                cen = np.clip(D_prev, prim_range[0] + 1e-12,
                                       prim_range[1] - 1e-12)
                lo_n = max(cen - hw, prim_range[0])
                hi_n = min(cen + hw, prim_range[1])
                roots = _find_roots(prim_func, lo_n, hi_n, z, prim_args)
                if not roots:
                    roots = _find_roots(prim_func, prim_range[0],
                                        prim_range[1], z, prim_args)
                if roots:
                    dists = [abs(r - D_prev) for r in roots]
                    D_sol = roots[np.argmin(dists)]
                else:
                    # Primary branch vanished → jump!
                    on_primary = False
                    roots = _find_roots(sec_func, sec_range[0],
                                        sec_range[1], z, sec_args)
                    if roots:
                        D_sol = roots[-1] if is_approach else roots[0]
                    else:
                        D_sol = D_prev
            else:
                hw = 3e-9
                cen = np.clip(D_prev, sec_range[0] + 1e-12,
                                       sec_range[1] - 1e-12)
                lo_n = max(cen - hw, sec_range[0])
                hi_n = min(cen + hw, sec_range[1])
                roots = _find_roots(sec_func, lo_n, hi_n, z, sec_args)
                if not roots:
                    roots = _find_roots(sec_func, sec_range[0],
                                        sec_range[1], z, sec_args)
                if roots:
                    dists = [abs(r - D_prev) for r in roots]
                    D_sol = roots[np.argmin(dists)]
                else:
                    D_sol = D_prev

            D_out.append(D_sol)
            # Standard AFM sign: F = k*(D − z), positive = repulsive
            F_out.append(k * (D_sol - z))
            D_prev = D_sol

        return z_array, np.array(D_out), np.array(F_out), D_prev

    # ── run both branches ─────────────────────────────────────────
    z_a, D_app, F_app, D_last = _solve(z_approach, "approach", z_approach[0])
    z_r, D_ret, F_ret, _      = _solve(z_retract, "retract",  D_last)
    return (z_a, F_app), (z_r, F_ret)

🔷 6. Define Baseline Parameters

In [ ]:
params = dict(
    k=0.2,                    # N/m — cantilever spring constant
    epsilon=2e-20,            # J   — LJ well depth
    sigma=0.35e-9,            # m   — LJ equilibrium distance
    r0=0.35e-9,               # m   — rest-position offset
    E_star=1e9,               # Pa  — reduced Young's modulus (1 GPa)
    R=20e-9,                  # m   — tip apex radius
    Wadh=50e-3,               # J/m² — work of adhesion
    adh_model="DMT",          #      — DMT or JKR
    hyst_strength=2e-9,       # N   — retraction hysteresis amplitude
    hyst_delta0=2e-9,         # m   — hysteresis saturation depth
)

🔷 7. Interactive Force–Distance Simulation

Use the sliders below to explore how each physical parameter affects the force–distance curve in real time. The plot updates automatically and displays the pull-off force and dissipated energy.

In [ ]:
# ── Interactive sliders ──────────────────────────────────────────────
style  = {"description_width": "160px"}
layout = widgets.Layout(width="480px")

slider_k = widgets.FloatLogSlider(
    value=0.2, base=10, min=-2, max=1, step=0.1,
    description="Cantilever k (N/m)", style=style, layout=layout,
    readout_format=".3f"
)
slider_E = widgets.FloatLogSlider(
    value=1e9, base=10, min=8, max=11, step=0.25,
    description="E* (Pa)", style=style, layout=layout,
    readout_format=".2e"
)
slider_R = widgets.FloatSlider(
    value=20, min=5, max=100, step=5,
    description="Tip radius R (nm)", style=style, layout=layout
)
slider_Wadh = widgets.FloatSlider(
    value=50, min=5, max=200, step=5,
    description="W_adh (mJ/m²)", style=style, layout=layout
)
slider_hyst = widgets.FloatSlider(
    value=2.0, min=0, max=10, step=0.5,
    description="Hysteresis (nN)", style=style, layout=layout
)
dropdown_model = widgets.Dropdown(
    options=["DMT", "JKR"],
    value="DMT",
    description="Adhesion model", style=style, layout=layout
)

output_fd = widgets.Output()

def update_fd_curve(change=None):
    # Build params dict from slider values
    p = dict(
        k=slider_k.value,
        epsilon=2e-20,
        sigma=0.35e-9,
        r0=0.35e-9,
        E_star=slider_E.value,
        R=slider_R.value * 1e-9,
        Wadh=slider_Wadh.value * 1e-3,
        adh_model=dropdown_model.value,
        hyst_strength=slider_hyst.value * 1e-9,
        hyst_delta0=2e-9,
    )

    (z_a, F_a), (z_r, F_r) = simulate_fd_curve(p, n=500)

    # Pull-off = most negative force during retraction (adhesion minimum)
    pull_off = np.min(F_r) * 1e9
    # Dissipated energy = area enclosed by the hysteresis loop
    E_app = np.trapezoid(F_a, z_a)
    E_ret = np.trapezoid(F_r, z_r)
    E_diss = abs(E_ret - E_app)

    with output_fd:
        clear_output(wait=True)
        fig, ax = plt.subplots(figsize=(8, 4.5))
        ax.plot(z_a * 1e9, F_a * 1e9, color="#2980B9", lw=2, label="Approach")
        ax.plot(z_r * 1e9, F_r * 1e9, color="#E74C3C", lw=2, label="Retraction")
        ax.axhline(0, color="grey", lw=0.5)
        ax.axvline(0, color="grey", lw=0.5, ls=":")
        ax.set_xlabel("Piezo position z (nm)")
        ax.set_ylabel("Force (nN)")
        ax.set_title("Simulated AFM Force–Distance Curve")
        ax.legend(loc="upper right")

        info = (f"Pull-off: {pull_off:.2f} nN  |  "
                f"E_diss: {E_diss:.2e} J  |  "
                f"Model: {dropdown_model.value}")
        ax.text(0.02, 0.98, info, transform=ax.transAxes,
                fontsize=9, ha="left", va="top",
                bbox=dict(facecolor="white", edgecolor="#ccc", alpha=0.85))
        plt.tight_layout()
        plt.show()

# Connect all widgets to the update function
for w in [slider_k, slider_E, slider_R, slider_Wadh, slider_hyst, dropdown_model]:
    w.observe(update_fd_curve, names="value")

controls = widgets.VBox([
    widgets.HBox([slider_k, slider_E]),
    widgets.HBox([slider_R, slider_Wadh]),
    widgets.HBox([slider_hyst, dropdown_model]),
])

display(controls, output_fd)
update_fd_curve()  # initial plot

In [ ]:
# ── Baseline run with default parameters (used by analysis cells below) ──
(z_app, F_app), (z_ret, F_ret) = simulate_fd_curve(params, n=500)

plt.figure(figsize=(8, 4.5))
plt.plot(z_app*1e9, F_app*1e9, color="#2980B9", lw=2, label="Approach")
plt.plot(z_ret*1e9, F_ret*1e9, color="#E74C3C", lw=2, label="Retraction")
plt.axhline(0, color="grey", lw=0.5)
plt.axvline(0, color="grey", lw=0.5, ls=":")
plt.xlabel("Piezo position z (nm)")
plt.ylabel("Force (nN)")
plt.title("Simulated AFM Force–Distance Curve (baseline parameters)")
plt.legend()
plt.tight_layout()
plt.show()

🔷 8. Extract Pull-Off Force and Dissipated Energy

In [ ]:
# Pull-off force: most negative value during retraction (adhesion)
pull_off = np.min(F_ret) * 1e9

# Dissipated energy: area enclosed by the hysteresis loop
E_app = np.trapezoid(F_app, z_app)
E_ret = np.trapezoid(F_ret, z_ret)
E_diss = abs(E_ret - E_app)

print(f"Pull-off force: {pull_off:.3f} nN")
print(f"Dissipated energy: {E_diss:.3e} J")

🔷 9. Hertz Fit to Extract Young’s Modulus

In [ ]:
def fit_hertz(z, F, k, R, Wadh, adh_model="DMT"):
    """Extract E* from the contact (indentation) region of an approach curve.

    Reconstructs indentation from z and F using D = z + F/k (from F = k(D-z)),
    then fits the Hertz model to the repulsive part of the curve.

    In the new sign convention (F = k*(D-z), positive = repulsive):
      F_measured = |F_adh| - F_Hertz   (in the contact region, delta > sigma)
    So: F_Hertz = |F_adh| - F_measured
    """
    # Reconstruct tip-sample distance: D = z + F/k  (from F = k*(D - z))
    D = z + F / k
    # Select the contact region (D < 0 means indentation)
    mask = D < 0
    delta = -D[mask]
    F_meas = F[mask]

    # Full adhesion magnitude (positive number)
    F_adh_mag = abs(adhesive_offset(R, Wadh, model=adh_model))

    # Extract Hertz contribution: F_H = F_adh_mag - F_measured
    # (because F_measured = F_adh_mag - F_H in the contact regime)
    F_hertz = F_adh_mag - F_meas

    # Only fit where Hertz is clearly positive and indentation is sufficient
    valid = (delta > 0.5e-9) & (F_hertz > 0)
    delta = delta[valid]
    F_hertz = F_hertz[valid]

    if len(delta) == 0:
        print("Warning: no valid contact data for Hertz fit")
        return np.nan

    X = delta**1.5
    Y = F_hertz

    A = np.sum(X * Y) / np.sum(X * X)
    E_star_fit = (3/4) * A / np.sqrt(R)
    return E_star_fit

E_fit = fit_hertz(z_app, F_app, params["k"], params["R"],
                  params["Wadh"], params["adh_model"])

print(f"Input E*: {params['E_star']:.2e} Pa")
print(f"Fitted E*: {E_fit:.2e} Pa")

🔷 10. Interactive Parameter Exploration

Compare how different values of a chosen parameter affect the approach curve. Select a parameter and adjust its range to see overlaid curves.

In [ ]:
# ── Interactive parameter comparison ─────────────────────────────────
param_select = widgets.Dropdown(
    options=[
        ("Cantilever k (N/m)", "k"),
        ("Tip radius R (nm)", "R"),
        ("Work of adhesion W_adh (mJ/m²)", "Wadh"),
        ("Reduced modulus E* (Pa)", "E_star"),
        ("Hysteresis strength (nN)", "hyst_strength"),
    ],
    value="k",
    description="Sweep parameter",
    style={"description_width": "130px"},
    layout=widgets.Layout(width="420px"),
)

# Default sweep ranges for each parameter (3 values each, in display units)
sweep_defaults = {
    "k":             {"values": [0.05, 0.2, 1.0],        "unit": "N/m",   "scale": 1},
    "R":             {"values": [10, 20, 50],             "unit": "nm",    "scale": 1e-9},
    "Wadh":          {"values": [10, 50, 150],            "unit": "mJ/m²", "scale": 1e-3},
    "E_star":        {"values": [1e8, 1e9, 1e11],         "unit": "Pa",    "scale": 1},
    "hyst_strength": {"values": [0.5, 2.0, 8.0],          "unit": "nN",    "scale": 1e-9},
}

sweep_v1 = widgets.FloatText(description="Value 1", style={"description_width": "60px"}, layout=widgets.Layout(width="180px"))
sweep_v2 = widgets.FloatText(description="Value 2", style={"description_width": "60px"}, layout=widgets.Layout(width="180px"))
sweep_v3 = widgets.FloatText(description="Value 3", style={"description_width": "60px"}, layout=widgets.Layout(width="180px"))

def set_defaults(change=None):
    d = sweep_defaults[param_select.value]["values"]
    sweep_v1.value, sweep_v2.value, sweep_v3.value = d[0], d[1], d[2]

param_select.observe(set_defaults, names="value")
set_defaults()

output_explore = widgets.Output()

def update_exploration(btn=None):
    key = param_select.value
    info = sweep_defaults[key]
    scale = info["scale"]
    unit  = info["unit"]
    values = [sweep_v1.value, sweep_v2.value, sweep_v3.value]

    base = dict(
        k=0.2, epsilon=2e-20, sigma=0.35e-9, r0=0.35e-9,
        E_star=1e9, R=20e-9, Wadh=50e-3, adh_model="DMT",
        hyst_strength=2e-9, hyst_delta0=2e-9,
    )

    colors = ["#2980B9", "#E74C3C", "#27AE60"]

    with output_explore:
        clear_output(wait=True)
        fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))

        for val, col in zip(values, colors):
            p = base.copy()
            p[key] = val * scale
            (z_a, F_a), (z_r, F_r) = simulate_fd_curve(p, n=500)

            label = f"{key}={val} {unit}"
            axes[0].plot(z_a * 1e9, F_a * 1e9, color=col, lw=2, label=label)
            axes[1].plot(z_a * 1e9, F_a * 1e9, color=col, lw=2, alpha=0.4)
            axes[1].plot(z_r * 1e9, F_r * 1e9, color=col, lw=2, ls="--", label=label)

        for ax in axes:
            ax.axhline(0, color="grey", lw=0.5)
            ax.axvline(0, color="grey", lw=0.5, ls=":")
            ax.set_xlabel("Piezo position z (nm)")
            ax.set_ylabel("Force (nN)")

        axes[0].set_title("Approach curves")
        axes[0].legend(fontsize=9)
        axes[1].set_title("Approach (solid) & Retraction (dashed)")
        axes[1].legend(fontsize=9)

        plt.tight_layout()
        plt.show()

run_btn = widgets.Button(description="Run comparison", button_style="primary",
                         layout=widgets.Layout(width="160px"))
run_btn.on_click(update_exploration)

display(
    widgets.HBox([param_select, run_btn]),
    widgets.HBox([sweep_v1, sweep_v2, sweep_v3]),
    output_explore,
)
update_exploration()

🔷 11. Discussion Questions
Why does jump-to-contact depend on cantilever stiffness?
Why does adhesion differ between DMT and JKR-inspired models?
Why does incorrect contact-point identification bias modulus extraction?
What physical processes could the hysteresis term represent?

🔷 12. Core Conceptual Message
A force–distance curve is the measurable outcome of a coupled system:
Atomic interaction + Surface thermodynamics + Contact mechanics + Cantilever dynamics.
Quantitative AFM requires identifying which regime dominates and which physical model applies.
Precision without conceptual awareness leads to systematic error.